# Part 5: Dashboard
Exports small aggregates and the compressed Phase 3 models into `dashboard.duckdb`, then writes the Streamlit app (overview, map, trip estimator, forecast). Live at [xzhou-c-nyc-taxi-streamlit-app-7zgufr.streamlit.app](https://xzhou-c-nyc-taxi-streamlit-app-7zgufr.streamlit.app/).

Part of [NYC Yellow Cab: demand, pricing and forecasting](../README.md).

In [16]:
# setup
import os, sys, json, gzip, shutil
from pathlib import Path
import duckdb, pandas as pd, numpy as np
import geopandas as gpd

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

BASE = Path(os.environ.get("NYC_TAXI_BASE", "/content/drive/MyDrive/nyc-yellow-cab"))
MODELS = BASE / "models"
APP = BASE / "dashboard"                 # the deployable app folder: code + data/
DATA = APP / "data"
(DATA / "models").mkdir(parents=True, exist_ok=True)
(APP / "app_pages").mkdir(exist_ok=True)
con = duckdb.connect(str(BASE / "taxi.duckdb"), read_only=True)
os.chdir(APP)                            # the %%writefile cells below write into the app folder
print("App folder:", APP)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
App folder: /content/drive/MyDrive/nyc-yellow-cab/dashboard


In [2]:
# 5.1 Zones
gdf = gpd.read_file(next((BASE / "raw" / "taxi_zones").rglob("*.shp")))
gdf.columns = [c.lower() for c in gdf.columns]
gdf = gdf.to_crs(2263).dissolve(by="locationid").reset_index()       # NY State Plane, feet; merge multi-part zones
cent = gdf.centroid

# A zone is inside the charge zone if trips that start and end in it are flagged touches_cbd (same rule as Phase 1)
cbd = con.sql("""SELECT pulocationid AS locationid FROM trips WHERE pulocationid = dolocationid
                 GROUP BY 1 HAVING count(*) >= 50 AND avg(touches_cbd::INT) > 0.99""").df()
zones = (con.sql("SELECT LocationID AS locationid, Zone AS zone, Borough AS borough FROM zones WHERE LocationID <= 263").df()
         .merge(pd.DataFrame({"locationid": gdf.locationid, "x_mi": cent.x / 5280, "y_mi": cent.y / 5280}),
                on="locationid", how="inner"))                      # same centroids the Phase 3 models were trained on
zones["in_cbd"] = zones.locationid.isin(cbd.locationid)
zones["is_airport"] = zones.locationid.isin([1, 132, 138])

shapes = gdf.assign(geometry=gdf.simplify(150))[["locationid", "geometry"]].to_crs(4326)   # 150 ft tolerance
# Plotly's map draws a polygon's inside only when its outer ring runs clockwise, so set that explicitly
from shapely.geometry import MultiPolygon
from shapely.geometry.polygon import orient
def clockwise(g):
    return orient(g, -1) if g.geom_type == "Polygon" else MultiPolygon([orient(p, -1) for p in g.geoms])
shapes["geometry"] = shapes.geometry.apply(clockwise)
shapes.to_file(DATA / "taxi_zones.geojson", driver="GeoJSON", COORDINATE_PRECISION=5)
print(f"{len(zones)} zones, {zones.in_cbd.sum()} inside the charge zone, "
      f"map file {(DATA / 'taxi_zones.geojson').stat().st_size / 1e6:.1f} MB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

263 zones, 23 inside the charge zone, map file 0.2 MB


In [3]:
# 5.2 Overview and map
daily = con.sql("SELECT * FROM daily_summary ORDER BY date").df()

# Average pickups per hour for each zone × weekday × hour. demand_zone_hour is zero-filled, so quiet hours count.
zone_profile = con.sql("""
    SELECT zone AS locationid, dayofweek(hour_ts) AS dow, hour(hour_ts) AS hour, avg(pickups) AS avg_pickups
    FROM demand_zone_hour GROUP BY ALL""").df()

# Weekly pickups per zone for the trend chart (only full weeks)
zone_weekly = con.sql("""
    WITH w AS (SELECT zone AS locationid, date_trunc('week', hour_ts)::DATE AS week,
                      sum(pickups) AS pickups, count(*) AS hours
               FROM demand_zone_hour GROUP BY ALL)
    SELECT locationid, week, pickups FROM w WHERE hours = 168 ORDER BY locationid, week""").df()
print(f"daily {len(daily):,} rows, zone_profile {len(zone_profile):,}, zone_weekly {len(zone_weekly):,}")

daily 730 rows, zone_profile 44,184, zone_weekly 27,352


In [4]:
# 5.3 Forecast table
forecast = con.sql("SELECT * FROM forecast_zone_hour ORDER BY unique_id, ds").df()
start = forecast.ds.min()
con.register("fc_ids", forecast[["unique_id"]].drop_duplicates())
actuals = con.sql(f"""
    WITH m AS (
      SELECT z.LocationID AS zone,
             CASE WHEN ('z' || z.LocationID) IN (SELECT unique_id FROM fc_ids) THEN 'z' || z.LocationID
                  ELSE 'other_' || z.Borough END AS unique_id
      FROM zones z WHERE z.LocationID <= 263)
    SELECT m.unique_id, d.hour_ts AS ds, sum(d.pickups)::DOUBLE AS y
    FROM demand_zone_hour d JOIN m USING (zone)
    WHERE d.hour_ts >= TIMESTAMP '{start}' - INTERVAL 14 DAY AND d.hour_ts < TIMESTAMP '{start}'
    GROUP BY ALL ORDER BY 1, 2""").df()
backtest = json.loads((MODELS / "forecast_backtest.json").read_text())
assert set(actuals.unique_id) == set(forecast.unique_id), "forecast series and actuals don't line up"
print(f"{forecast.unique_id.nunique()} series, forecast from {start:%a %b %d}, best model: {backtest['best_model']}")

56 series, forecast from Tue Sep 01, best model: LightGBM (post-Jan 2025 data)


In [6]:
# 5.4 Phase 3 models
for name in ["duration", "fare", "tip_pct"]:
    with open(MODELS / f"{name}.txt", "rb") as src, gzip.open(DATA / "models" / f"{name}.txt.gz", "wb") as dst:
        shutil.copyfileobj(src, dst)
    print(f"{name}: {(MODELS / f'{name}.txt').stat().st_size / 1e6:.1f} MB → "
          f"{(DATA / 'models' / f'{name}.txt.gz').stat().st_size / 1e6:.1f} MB")
test_metrics = json.loads((MODELS / "test_metrics.json").read_text())
_ = (DATA / "metrics.json").write_text(json.dumps({"models": test_metrics, "forecast": backtest}, indent=2, default=str))

duration: 30.3 MB → 11.2 MB
fare: 44.0 MB → 16.2 MB
tip_pct: 42.8 MB → 15.7 MB


In [7]:
# 5.5 DuckDB file
out = DATA / "dashboard.duckdb"
out.unlink(missing_ok=True)
db = duckdb.connect(str(out))
for name, frame in {"zones": zones, "daily_summary": daily, "zone_profile": zone_profile, "zone_weekly": zone_weekly,
                    "forecast": forecast, "forecast_actuals": actuals}.items():
    db.register("tmp", frame)
    db.execute(f"CREATE TABLE {name} AS SELECT * FROM tmp")
    db.unregister("tmp")
db.close()
print(f"dashboard.duckdb: {out.stat().st_size / 1e6:.1f} MB")

dashboard.duckdb: 2.1 MB


In [8]:
%%writefile requirements.txt
streamlit>=1.50
duckdb>=1.1
pandas>=2.2
plotly>=5.24
lightgbm>=4.3
holidays>=0.50

Writing requirements.txt


In [9]:
%%writefile streamlit_app.py
import streamlit as st

st.set_page_config(page_title="NYC Yellow Cab", page_icon="🚕", layout="wide")
pages = st.navigation([
    st.Page("app_pages/overview.py", title="Overview", icon=":material/insights:", default=True),
    st.Page("app_pages/map.py", title="Map", icon=":material/map:"),
    st.Page("app_pages/estimator.py", title="Trip estimator", icon=":material/local_taxi:"),
    st.Page("app_pages/forecast.py", title="Forecast", icon=":material/trending_up:"),
])
pages.run()

Writing streamlit_app.py


In [10]:
%%writefile common.py
import gzip, json
from pathlib import Path
import duckdb, pandas as pd, streamlit as st

DATA = Path(__file__).parent / "data"
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GRAY = "#b5b4ae"
DOW = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]          # DuckDB dayofweek: 0 = Sunday


@st.cache_resource
def _db():
    return duckdb.connect(str(DATA / "dashboard.duckdb"), read_only=True)


@st.cache_data
def q(sql, params=None):
    with _db().cursor() as cur:                                   # one cursor per query: safe across sessions
        return cur.execute(sql, params or []).df()


@st.cache_data
def zones():
    z = q("SELECT * FROM zones ORDER BY zone")
    z["name"] = z.zone + " (" + z.borough + ")"
    return z


@st.cache_data
def geojson():
    return json.loads((DATA / "taxi_zones.geojson").read_text())


@st.cache_data
def metrics():
    return json.loads((DATA / "metrics.json").read_text())


@st.cache_resource
def booster(name):
    import lightgbm as lgb
    with gzip.open(DATA / "models" / f"{name}.txt.gz", "rt") as f:
        return lgb.Booster(model_str=f.read())


def style(fig, height=320):
    fig.update_layout(template="plotly_white", height=height, margin=dict(l=10, r=10, t=40, b=40),
                      colorway=C, font=dict(size=13), title_font=dict(size=16),
                      legend=dict(orientation="h", yanchor="top", y=-0.12, x=0, title=None))
    fig.update_xaxes(showgrid=False, title=None)
    return fig

Writing common.py


In [11]:
%%writefile app_pages/overview.py
import pandas as pd, plotly.express as px, plotly.graph_objects as go, streamlit as st
from common import q, style, C, GRAY

st.title("NYC yellow cab trips")
d = q("SELECT * FROM daily_summary ORDER BY date")
d["date"] = pd.to_datetime(d.date)
st.caption(f"{d.trips.sum() / 1e6:,.1f} million clean trips, {d.date.min():%b %Y} to {d.date.max():%b %Y}. "
           "Source: NYC TLC trip records.")

# Monthly figures. Average fare is weighted by trips, so it equals total fares / total trips for the month.
m = (d.assign(month=d.date.dt.to_period("M"), fare_x_trips=d.avg_fare * d.trips)
       .groupby("month").agg(days=("date", "size"), trips=("trips", "sum"), revenue=("revenue", "sum"),
                             fare_x_trips=("fare_x_trips", "sum"), tip=("median_tip_pct_card", "mean")))
m["trips_per_day"], m["revenue_per_day"] = m.trips / m.days, m.revenue / m.days
m["avg_fare"] = m.fare_x_trips / m.trips

months = [str(p) for p in m.index]
pick = st.select_slider("Month", options=months, value=months[-1],
                        format_func=lambda s: pd.Period(s).strftime("%b %Y"))
cur = m.loc[pd.Period(pick)]
prev = m.shift(1).loc[pd.Period(pick)]


def delta(a, b, fmt):
    return None if pd.isna(a) or pd.isna(b) else fmt(a - b)


k = st.columns(4)
k[0].metric("Trips per day", f"{cur.trips_per_day:,.0f}",
            delta(cur.trips_per_day / prev.trips_per_day * 100, 100, lambda x: f"{x:+.1f}% vs. prior month"))
k[1].metric("Revenue per day", f"${cur.revenue_per_day / 1e6:,.2f}M",
            delta(cur.revenue_per_day / prev.revenue_per_day * 100, 100, lambda x: f"{x:+.1f}%"))
k[2].metric("Average fare", f"${cur.avg_fare:,.2f}", delta(cur.avg_fare, prev.avg_fare, lambda x: f"{x:+.2f}"))
k[3].metric("Median tip (card trips)", f"{cur.tip:.1f}%", delta(cur.tip, prev.tip, lambda x: f"{x:+.1f} pts"))

d["avg7"] = d.trips.rolling(7, center=True).mean()
fig = go.Figure()
fig.add_scatter(x=d.date, y=d.trips, mode="lines", line=dict(color=GRAY, width=1), name="Daily")
fig.add_scatter(x=d.date, y=d.avg7, mode="lines", line=dict(color=C[0], width=2.5), name="7-day average")
sel = pd.Period(pick)
fig.add_vrect(x0=sel.start_time, x1=sel.end_time, fillcolor=C[0], opacity=0.08, line_width=0)
fig.update_yaxes(title="Trips per day")
st.plotly_chart(style(fig.update_layout(title="Trips per day"), 340))

left, right = st.columns(2)
mm = m.reset_index().assign(month=lambda x: x.month.dt.to_timestamp())
f2 = px.bar(mm, x="month", y="revenue_per_day", title="Revenue per day, by month", color_discrete_sequence=[C[0]])
f2.update_yaxes(title=None, tickprefix="$", rangemode="tozero")
left.plotly_chart(style(f2, 300))
f3 = px.line(mm, x="month", y="avg_fare", markers=True, title="Average fare, by month", color_discrete_sequence=[C[1]])
f3.update_yaxes(title=None, tickprefix="$")
right.plotly_chart(style(f3, 300))

Writing app_pages/overview.py


In [12]:
%%writefile app_pages/map.py
import numpy as np, pandas as pd, plotly.express as px, streamlit as st
from common import q, zones, geojson, style, C, DOW

st.title("Where the pickups are")
z = zones()
c1, c2 = st.columns([1, 2])
days = c1.segmented_control("Days", ["All days", "Weekdays", "Weekends"], default="All days")
h0, h1 = c2.slider("Pickup hour", 0, 23, (0, 23), format="%d:00")
dows = {"All days": range(7), "Weekdays": range(1, 6), "Weekends": (0, 6)}[days or "All days"]

prof = q("SELECT * FROM zone_profile")
win = prof[prof.dow.isin(list(dows)) & prof.hour.between(h0, h1)]
by_zone = (win.groupby("locationid").avg_pickups.mean().rename("per_hour").reset_index()
             .merge(z[["locationid", "zone", "borough"]], on="locationid"))
by_zone["log"] = np.log10(by_zone.per_hour.clip(lower=0.1))       # log color scale, labeled in real numbers

ticks = [0.1, 1, 10, 100, 1000]
# Drawn as SVG with no basemap tiles, so it loads fast and works anywhere
fig = px.choropleth(by_zone, geojson=geojson(), locations="locationid", featureidkey="properties.locationid",
                    color="log", color_continuous_scale="Blues", range_color=(-1, np.log10(max(by_zone.per_hour.max(), 1))),
                    hover_name="zone", hover_data={"borough": True, "per_hour": ":.1f", "log": False, "locationid": False},
                    labels={"per_hour": "Pickups per hour"})
fig.update_geos(fitbounds="locations", visible=False)
fig.update_traces(marker_line_color="white", marker_line_width=0.4)
fig.update_coloraxes(colorbar=dict(title="Pickups<br>per hour", tickvals=np.log10(ticks), ticktext=["0.1", "1", "10", "100", "1,000"]))
fig.update_layout(height=600, margin=dict(l=0, r=0, t=0, b=0), paper_bgcolor="rgba(0,0,0,0)", geo_bgcolor="rgba(0,0,0,0)")
st.plotly_chart(fig)
st.caption(f"Average pickups per hour, {days.lower() if days else 'all days'}, {h0}:00–{h1}:59. "
           "Hover a zone for its number; pick one below for its trend.")

default = int(by_zone.sort_values("per_hour").locationid.iloc[-1])      # the busiest zone in the current filter
names = z.set_index("locationid")["name"]
zone_id = st.selectbox("Zone", z.locationid, index=int(np.flatnonzero(z.locationid == default)[0]),
                       format_func=lambda i: names[i])

a, b = st.columns(2)
wk = q("SELECT week, pickups FROM zone_weekly WHERE locationid = ? ORDER BY week", [int(zone_id)])
f1 = px.line(wk, x="week", y="pickups", title=f"Weekly pickups: {names[zone_id]}", color_discrete_sequence=[C[0]])
f1.update_yaxes(title=None, rangemode="tozero")
a.plotly_chart(style(f1))

zp = prof[prof.locationid == zone_id].assign(day=lambda x: np.where(x.dow.isin([0, 6]), "Weekend", "Weekday"))
zp = zp.groupby(["day", "hour"]).avg_pickups.mean().reset_index()
f2 = px.line(zp, x="hour", y="avg_pickups", color="day", title="Average pickups by hour",
             color_discrete_map={"Weekday": C[0], "Weekend": C[1]})
f2.update_xaxes(tickvals=list(range(0, 24, 3)), ticktext=[f"{h}:00" for h in range(0, 24, 3)])
f2.update_yaxes(title=None, rangemode="tozero")
b.plotly_chart(style(f2))

Writing app_pages/map.py


In [13]:
%%writefile app_pages/estimator.py
import datetime as dt
import holidays, numpy as np, pandas as pd, plotly.express as px, streamlit as st
from common import zones, booster, metrics, style, C

st.title("Trip estimator")
st.caption("Predicted from what's known at booking: pickup and drop-off zone and time. Standard metered trips.")
z = zones().set_index("locationid")
ids = list(z.index)
default_pu = 161 if 161 in ids else ids[0]
default_do = 132 if 132 in ids else ids[-1]

c = st.columns([2, 2, 1, 1])
pu = c[0].selectbox("Pickup zone", ids, index=ids.index(default_pu), format_func=lambda i: z["name"][i])
do = c[1].selectbox("Drop-off zone", ids, index=ids.index(default_do), format_func=lambda i: z["name"][i])
day = c[2].date_input("Date", dt.date.today() + dt.timedelta(days=1))
t = c[3].time_input("Pickup time", dt.time(17, 0), step=1800)
NY_HOLIDAYS = holidays.US(state="NY", years=range(2024, dt.date.today().year + 3))


def features(hours):
    """One row per pickup hour, built the same way as the Phase 3 modeling table."""
    a, b = z.loc[pu], z.loc[do]
    dow = (day.weekday() + 1) % 7                                  # 0 = Sunday, as in DuckDB
    f = pd.DataFrame({"pu_zone_id": pu, "do_zone_id": do, "pu_hour": hours, "pu_dow": dow, "month": day.month,
                      "is_weekend": int(dow in (0, 6)), "is_holiday": int(day in NY_HOLIDAYS),
                      "is_airport": int(a.is_airport or b.is_airport), "touches_cbd": int(a.in_cbd or b.in_cbd),
                      "post_cp": int(day >= dt.date(2025, 1, 5)), "pu_x": a.x_mi, "pu_y": a.y_mi, "do_x": b.x_mi,
                      "do_y": b.y_mi, "centroid_miles": np.hypot(a.x_mi - b.x_mi, a.y_mi - b.y_mi),
                      "same_zone": int(pu == do), "passengers": 1})
    for col in ["pu_zone_id", "do_zone_id"]:
        f[col] = f[col].astype("category")                         # LightGBM maps these to its stored categories
    return f


def predict(f):
    dur, fare, tip = booster("duration"), booster("fare"), booster("tip_pct")
    minutes = np.exp(dur.predict(f[dur.feature_name()]))           # duration model is trained on log minutes
    f = f.assign(fare_amount=fare.predict(f[fare.feature_name()]).clip(min=3))
    tip_pct = tip.predict(f[tip.feature_name()]).clip(0, 100)
    return minutes, f.fare_amount.to_numpy(), tip_pct


day_rows = features(list(range(24)))
minutes, fare, tip_pct = predict(day_rows)
h = t.hour
k = st.columns(4)
k[0].metric("Trip time", f"{minutes[h]:.0f} min")
k[1].metric("Metered fare", f"${fare[h]:.2f}")
k[2].metric(f"Tip on a card ({tip_pct[h]:.0f}% of fare)", f"${fare[h] * tip_pct[h] / 100:.2f}")
k[3].metric("Straight-line distance", f"{day_rows.centroid_miles.iloc[0]:.1f} mi")

m = metrics()["models"]
mae = {w: next(v["model_MAE"] for k, v in m.items() if w in k) for w in ("Duration", "Fare")}   # match by name, whatever the prefix
st.caption(f"Typical error on unseen months: ±{mae['Duration']:.1f} min, "
           f"±${mae['Fare']:.2f} fare. Fare excludes tolls and surcharges.")

plan = pd.DataFrame({"hour": range(24), "minutes": minutes})
fig = px.line(plan, x="hour", y="minutes", markers=True, color_discrete_sequence=[C[0]],
              title=f"Predicted trip time by pickup hour on {day:%a %b %-d}")
fig.add_scatter(x=[h], y=[minutes[h]], mode="markers", marker=dict(size=13, color=C[1]), name="Your pickup time")
fig.update_xaxes(tickvals=list(range(0, 24, 3)), ticktext=[f"{x}:00" for x in range(0, 24, 3)])
fig.update_yaxes(title="Minutes", rangemode="tozero")
st.plotly_chart(style(fig, 340))

Writing app_pages/estimator.py


In [14]:
%%writefile app_pages/forecast.py
import pandas as pd, plotly.graph_objects as go, plotly.express as px, streamlit as st
from common import q, metrics, style, C, GRAY

st.title("Demand forecast: next 7 days")
fc = q("SELECT * FROM forecast ORDER BY unique_id, ds")
act = q("SELECT * FROM forecast_actuals ORDER BY unique_id, ds")
bt = metrics()["forecast"]
best = bt["best_model"]
st.caption(f"Hourly pickups forecast by {best}, made {pd.Timestamp(fc.made_at.iloc[0]):%b %-d, %Y}, "
           f"for {fc.ds.min():%a %b %-d} to {fc.ds.max():%a %b %-d}.")

labels = fc.groupby("unique_id").label.first().sort_values()
default = labels.index.get_loc("z161") if "z161" in labels.index else 0
uid = st.selectbox("Zone", labels.index, index=default, format_func=lambda u: labels[u])

f, a = fc[fc.unique_id == uid], act[act.unique_id == uid]
fig = go.Figure()
fig.add_scatter(x=a.ds, y=a.y, mode="lines", line=dict(color=GRAY, width=1.5), name="Last 14 days (actual)")
fig.add_scatter(x=f.ds, y=f.forecast, mode="lines", line=dict(color=C[0], width=2), name="Forecast")
fig.add_vline(x=f.ds.min().isoformat(), line_dash="dot", line_color="#52514e")
fig.update_yaxes(title="Pickups per hour", rangemode="tozero")
st.plotly_chart(style(fig.update_layout(title=labels[uid]), 360))

peak = f.loc[f.forecast.idxmax()]
last_week = a[a.ds >= f.ds.min() - pd.Timedelta(days=7)].y.sum()
k = st.columns(3)
k[0].metric("Pickups next 7 days", f"{f.forecast.sum():,.0f}",
            f"{(f.forecast.sum() / last_week - 1) * 100:+.0f}% vs. last 7 days" if last_week else None)
k[1].metric(f"Busiest hour ({peak.forecast:,.0f} pickups)", f"{peak.ds:%a %-I %p}")
k[2].metric(f"Backtest error, WAPE (same-hour-last-week: {bt['overall']['Seasonal naive']['WAPE']:.0%})",
            f"{bt['overall'][best]['WAPE']:.0%}")

st.subheader("How accurate is it?")
st.caption("Rolling-origin backtest: 12 times, the model forecast a week it hadn't seen. "
           "MASE below 1 beats the same-hour-last-week forecast. WAPE is the share of pickups mispredicted.")
left, right = st.columns([3, 2])
ov = pd.DataFrame(bt["overall"]).T[["MASE", "WAPE"]]
left.dataframe(ov.style.format({"MASE": "{:.2f}", "WAPE": "{:.1%}"}))
bb = pd.DataFrame(bt["by_borough_WAPE"]).T
bb = bb.loc[[b for b in ["Manhattan", "Queens", "Brooklyn", "Bronx"] if b in bb.index], [best]].reset_index()
f2 = px.bar(bb, x=best, y="index", orientation="h", title=f"{best}: WAPE by borough", color_discrete_sequence=[C[0]])
f2.update_xaxes(tickformat=".0%", title=None)
f2.update_yaxes(title=None, autorange="reversed")
right.plotly_chart(style(f2, 260))

Writing app_pages/forecast.py


In [15]:
import zipfile
files = sorted(p for p in APP.rglob("*") if p.is_file() and "__pycache__" not in p.parts)
size = sum(p.stat().st_size for p in files)
print("\n".join(f"{p.stat().st_size / 1e6:7.2f} MB  {p.relative_to(APP)}" for p in files))
print(f"{size / 1e6:7.2f} MB  total")
assert all(p.stat().st_size < 95e6 for p in files), "a file is over GitHub's 100 MB limit"

zip_path = BASE / "nyc_taxi_dashboard.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        zf.write(p, Path("nyc-taxi-dashboard") / p.relative_to(APP))
print(f"Saved {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
if "google.colab" in sys.modules:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))

   0.00 MB  app_pages/estimator.py
   0.00 MB  app_pages/forecast.py
   0.00 MB  app_pages/map.py
   0.00 MB  app_pages/overview.py
   0.00 MB  common.py
   2.11 MB  data/dashboard.duckdb
   0.00 MB  data/metrics.json
  11.22 MB  data/models/duration.txt.gz
  16.17 MB  data/models/fare.txt.gz
  15.69 MB  data/models/tip_pct.txt.gz
   0.20 MB  data/taxi_zones.geojson
   0.00 MB  requirements.txt
   0.00 MB  streamlit_app.py
  45.42 MB  total
Saved /content/drive/MyDrive/nyc-yellow-cab/nyc_taxi_dashboard.zip (43.4 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>